In [1]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline
import warnings
warnings.filterwarnings("ignore")


c:\big21\agentic_ai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\big21\agentic_ai\venv\lib\site-packages\pyannote\audio\core\io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


In [3]:
# ------------------------------------------------------------------
# 경로를 새로 다운받은 "shared" 폴더의 bin 경로로 교체해주세요!
# (예시 경로이므로 본인의 실제 다운로드 경로로 수정 필수)
# ------------------------------------------------------------------
ffmpeg_path = r"C:\big21\agentic_ai\chap05\ffmpeg-7.1.1-full_build-shared\bin"


if os.path.exists(ffmpeg_path):
    # 1. PATH 환경 변수 추가
    if ffmpeg_path not in os.environ.get('PATH', ''):
        os.environ['PATH'] = ffmpeg_path + os.pathsep + os.environ.get('PATH', '')
   
    # 2. DLL 디렉토리 추가 (가장 중요!)
    if hasattr(os, 'add_dll_directory'):
        try:
            os.add_dll_directory(ffmpeg_path)
            print("FFmpeg DLL 로딩 경로 설정 완료!")
        except Exception as e:
            print(f"설정 중 에러: {e}")


FFmpeg DLL 로딩 경로 설정 완료!


In [4]:
def whisper_to_dataframe(result, output_file_path):
    start_end_text=[]

    for chunk in result["chunks"]:
        start, end = chunk["timestamp"]
        text = chunk["text"].strip()    # 양쪽 공백 제거
        start_end_text.append([start,end,text]) # 데이터프레임 생성에 필요한 데이터 준비    
                                                # [[]]로 해서 2차원.(index, columns, values 중 values 넣어야 해서 2차원.)
    df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
    df.to_csv(output_file_path, index=False, sep="|")   # csv 생성

    return df

In [5]:
def whisper_stt(
        audio_file_path:str,        # 변환할 음성 파일(경로) 
        output_file_path:str = "output.csv"     # 최종 결과물이 아니라 중간 결과물이라 default 지정함..?
):
    # 1. 디바이스 지정
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    
    # 2. dtype 지정
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    
    # whisper model 지정
    model_id = "openai/whisper-large-v3-turbo"
    
    # speech seq
    model = AutoModelForSpeechSeq2Seq.from_pretrained(  # 모델이 캐시에 없으면 다운로드, 있으면 오브젝트 생성
        model_id,   # model name, path
        torch_dtype=torch_dtype,    # float16, 경고(torch_dtype is deprecated. Use 'dtype' instead of torch_dtype. 없어질 예정.)
        low_cpu_mem_usage=True,
        use_safetensors=True
    )
    model.to(device)
    # audio => model => feature_extractor => tokenizer
    # feature_extractor => tokenizer
    processor=AutoProcessor.from_pretrained(model_id)
    pipe=pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        dtype=torch_dtype,
        device=device,
        return_timestamps=True,      # 청크별로 타임스탬프 반환
        chunk_length_s=10,      # 입력 오디오를 10초씩 나누기
        stride_length_s=2       # 청크 2초씩 겹치도록 나누기
    )

    # STT: audio -> text
    result=pipe(audio_file_path)

    # result => dataframe function
    df=whisper_to_dataframe(result,output_file_path)

    return result, df

In [ ]:
def speaker_diarization(    # 화자 분리
        audio_file_path: str,
        output_rttm_file_path: str,
        output_csv_file_path: str
    ):
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        # use_auth_token
    )


    # cuda가 사용 가능한 경우 cuda를 사용하도록 설정
    if torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))
        print('cuda is available')
    else:
        print('cuda is not available')
    diarization_pipeline = pipeline(audio_file_path)


    # dump the diarization output to disk using RTTM format
    with open(output_rttm_file_path, "w", encoding='utf-8') as rttm:
        diarization_pipeline.write_rttm(rttm)


    # pandas dataframe으로 변환
    df_rttm = pd.read_csv(
        output_rttm_file_path,       # rttm 파일 경로
        sep=' ',        # 구분자는 띄어쓰기
        header=None,    # 헤더는 없음
        names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4']
    )
   
    df_rttm["end"] = df_rttm["start"] + df_rttm["duration"]


    # speaker_id를 기반으로 화자별로 구간을 나누기
    df_rttm["number"] = None
    df_rttm.at[0, "number"] = 0


    for i in range(1, len(df_rttm)):
        if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
            df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
        else:
            df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]


    df_rttm_grouped = df_rttm.groupby("number").agg(
        start=pd.NamedAgg(column='start', aggfunc='min'),
        end=pd.NamedAgg(column='end', aggfunc='max'),
        speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
    )


    df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]


    df_rttm_grouped.to_csv(
        output_csv_file_path,
        index=False,    # 인덱스는 저장하지 않음
        encoding='utf-8'
    )
    return df_rttm_grouped


In [ ]:
# 최종 결과 추출(whisper(whisper_stt func), 화자분리(speaker_diarization func) 등)
def stt_to_rttm(
        audio_file_path:str,    # whisper/화자분리 같이 사용됨.
        stt_output_file_path:str,   # whisper output 파일
        rttm_file_path:str,     # 화자 분리
        rttm_csv_file_path:str,
        final_output_csv_file_path:str
):
    # 1. STT : whisper_stt()
    result, df_stt = whisper_stt(
        audio_file_path=audio_file_path,
        output_file_path=stt_output_file_path
        )
    # 2. diarization : speaker_diarization()
    df_rttm = speaker_diarization(  # 화자 분리 결과 데이터
        audio_file_path=audio_file_path,
        # output_rttm_file_path(speaker_diarization 선언된 매개변수명)
        # audio_file_path(stt_to_rttm 선언된 매개변수명)
        output_rttm_file_path=rttm_file_path,
        output_csv_file_path=rttm_csv_file_path
    )
    # df_rttm dataframe에 text 저장 컬럼 선언
    df_rttm["text"] = ""
    # i_stt: 줄번호, row_stt: 한 행(start,end,text)
    #  
    for i_stt, row_stt in df_stt.iterrows(): # df_stt: dataframe. iterrows(): 행 단위로 하나씩 꺼내줘야 할 때 사용.
        overlap_dict = {}   # 겹치는 부분 저장하는 딕셔너리
        for i_rttm, row_rttm in df_rttm.iterrows(): # 화자 분리 , row_rttm: start,end,speaker_id
            # min(row_stt["end"], row_rttm["end"]): 두 끝 시간 중 빠른 쪽.
            # max(row_stt["start"], row_rttm["start"]): 두 시작 시간 중 늦은 쪽
            # 빠른 쪽 끝 - 늦은 시작 = 겹친 시간. 전혀 안 겹치면 음수. max로 처리하여 겹친 시간이 음수면 overlap=0
            overlap = max(0, min(row_stt["end"], row_rttm["end"]) - max(row_stt["start"], row_rttm["start"]))
            overlap_dict[i_rttm] = overlap
        # end for
        
        max_overlap = max(overlap_dict.values())
        max_overlap_idx = max(overlap_dict, key=overlap_dict.get)


        if max_overlap > 0: #
            df_rttm.at[max_overlap_idx, "text"] += row_stt["text"] + "\n"
    # end for
    df_rttm.to_csv(
        final_output_csv_file_path,
        index=False,    # 인덱스는 저장하지 않음
        sep='|',
        encoding='utf-8'
    )  #
    return df_rttm  # dataframe 반환

In [8]:
audio_file_path = "../audio/싼기타_비싼기타.mp3"       # 원본 오디오 파일
stt_output_file_path = "싼기타_비싼기타.csv"    # STT 결과 파일
rttm_file_path = "싼기타_비싼기타.rttm"     # 화자 분리 원본 파일
rttm_csv_file_path = "싼기타_비싼기타_rttm.csv" # 화자 분리 CSV 파일
final_csv_file_path = "싼기타_비싼기타_final.csv" # 최종 결과 파일

In [9]:
df_rttm = stt_to_rttm(
    audio_file_path,
    stt_output_file_path,
    rttm_file_path,
    rttm_csv_file_path,
    final_csv_file_path
)


print(df_rttm)


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See 

cuda is available
          start      end  speaker_id  duration  \
number                                           
0         0.976   30.195  SPEAKER_01    29.219   
1        32.419   41.638  SPEAKER_00     9.219   
2        41.638   42.453  SPEAKER_01     0.815   
3        41.655   42.725  SPEAKER_00     1.070   
4        42.674   44.032  SPEAKER_01     1.358   
5        45.798   67.105  SPEAKER_00    21.307   
6        67.224   82.793  SPEAKER_01    15.569   
7        84.643  102.572  SPEAKER_00    17.929   
8       103.489  117.530  SPEAKER_01    14.041   
9       119.737  138.667  SPEAKER_00    18.930   
10      139.346  168.956  SPEAKER_01    29.610   
11      170.925  192.318  SPEAKER_00    21.393   
12      192.317  193.692  SPEAKER_01     1.375   
13      192.776  193.523  SPEAKER_00     0.747   
14      193.812  216.579  SPEAKER_01    22.767   
15      218.565  237.767  SPEAKER_00    19.202   
16      238.107  238.667  SPEAKER_01     0.560   
17      238.175  239.346  SPEAKE